# Ocean sounds: evaluation of Path A (128-band mel)

Scores the synthetic clips from `ocean_path_a.ipynb` against real ocean recordings. Nothing is
trained here except one small classifier used as a judge and an embedder. The method is the one used
on ESC-50 (`evaluate_paths.ipynb`): everything is compared **as audio**, every distance is shown
against a **real-vs-real floor**, and the **ceiling** (real clips through the mel representation
and Griffin-Lim, no generator) separates what the representation costs from what the generator
adds.

What is different for this dataset, and why:

| Change | Reason |
|---|---|
| The real reference is the **validation fold (4)**, not fold 5 | Fold 5 is reserved for the final TSTR test. The pre-registered TSTR design computes every metric against fold 4, so the metrics never see the test data. |
| The reference is **class-balanced** to match the generated set | The generated set has the same number of clips per class; the real data is 8.6 : 1 imbalanced. Comparing the two directly would mostly measure the class mix, not the sound. |
| An **in-domain embedder** next to PANNs and CLAP | PANNs and CLAP were trained on terrestrial audio at 32/48 kHz. A small CNN trained on real ocean clips (the probe recipe) knows what separates three nearly identical groupers. It is the primary embedder here. OceanBEATs is a candidate to add later. |
| **Class fidelity** (train on real, test on synthetic) | The judge classifier labels each generated clip. If generated Nassau groupers are recognised as Nassau about as often as real clips after Griffin-Lim are, the class information survived generation. |
| **Per-class** distances | The fine-grained classes (three groupers, squirrelfish) are the headline; toadfish and vessel are separable by recording site, so their numbers say little about the calls. |
| **Learnability on validation clips** as well as training clips | The training notebook measured it on training clips only, which measures fit. The gap to validation clips measures generalization. |
| **Memorization check** | The data has many near-identical clips from the same recordings, which makes copying easier. Generated clips should be no closer to the training set than unseen real clips are. |

Runtime on a T4: about 15-20 minutes the first time (PANNs and CLAP downloads, judge training),
a few minutes after that (results are cached in `EVAL_DIR`). Set `QUICK_TEST = True` first.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile panns_inference "transformers>=4.45"
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import eval_metrics as M

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pathlib import Path
from IPython.display import Audio, display
from diffusers.training_utils import EMAModel

assert hasattr(C, "load_ocean"), "common.py is an old version: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

The data and front-end settings **must match the training notebook**. In particular `GROUP_SIZE`
decides which clips are in which fold: with a different value, the "validation" clips could
include clips the generator was trained on. The provenance check below stops the notebook if the
sample file was made with different settings.

In [ ]:
QUICK_TEST = False          # True: a few minutes end to end, numbers meaningless

# data and front end: must match notebooks/ocean_path_a.ipynb
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"
GROUP_SIZE  = 20
HIGHPASS_HZ = 20.0
NORMALIZE   = "rms"
BALANCED    = True
OUT_ROOT    = "/content/drive/MyDrive/thesis_runs/ocean"

cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data")
FE_TAG        = "" if (cfg.n_fft, cfg.f_max) == (1024, None) else f"_fft{cfg.n_fft}_fmax{cfg.f_max:g}"
TAG           = FE_TAG + f"_hp{HIGHPASS_HZ:g}_{NORMALIZE}" + ("_bal" if BALANCED else "")
RUN_DIR       = f"{cfg.out_root}/path_a128{TAG}"
FRONTEND_FILE = f"{cfg.out_root}/frontend_128mel{FE_TAG}_hp{HIGHPASS_HZ:g}_{NORMALIZE}.json"
SAMPLES_FILE  = f"{cfg.out_root}/samples/path_a128{TAG}.pt"
EVAL_DIR      = Path(f"{cfg.out_root}/eval/path_a128{TAG}" + ("_quick" if QUICK_TEST else ""))
EVAL_DIR.mkdir(parents=True, exist_ok=True)

FINE = ["Nassau grouper", "Yellowfin grouper", "Black grouper", "Squirrelfish"]

# reference set and metrics
REAL_PER_GEN  = 2            # real clips per class = 2 x generated clips per class (floor halves match the generated set)
EMBEDDERS     = ("probe", "panns", "clap")   # the first one is the primary
MAX_DIM       = 32           # PCA dimensions, all classes together
MAX_DIM_CLASS = 8            # PCA dimensions, one class at a time (~100 clips per half)
N_SPLITS, N_PERM = 30, 1000
GL_ITERS      = 64

# judge classifier (the probe recipe, one fixed seed: the same judge for every synthetic set later)
JUDGE_SEED, EPOCHS, BATCH, LR = 0, 6, 128, 2e-3

# learnability
LEARN_PER_CLASS = 64         # clips per class, from the training folds and from the validation fold
UNET_BATCH      = 64         # clips per diffusion-model forward pass (lower it if the GPU runs out of memory)

GEN_PER_CLASS = None         # None: every generated clip in the file
if QUICK_TEST:
    N_SPLITS, N_PERM, GL_ITERS, EPOCHS, LEARN_PER_CLASS, GEN_PER_CLASS = 8, 100, 8, 1, 8, 10
C.seed_everything(cfg.seed)
print("samples :", SAMPLES_FILE)
print("run dir :", RUN_DIR)
print("eval dir:", EVAL_DIR)

## 3. Provenance check

Reads the settings stored with the sample file and stops if they differ from this notebook's. A
mismatch in the data settings would mean comparing against the wrong real data; a mismatch in the
front end would make the Griffin-Lim inversion wrong.

In [ ]:
s = C.load_samples(SAMPLES_FILE)
meta, scfg = s["meta"], s["config"]
problems = []
for k in ("sample_rate", "n_fft", "hop_length", "n_mels", "f_min", "f_max", "spec_frames", "db_range", "val_fold", "heldout_fold"):
    if scfg.get(k) != getattr(cfg, k):
        problems.append(f"config {k}: sample file {scfg.get(k)} vs this notebook {getattr(cfg, k)}")
for k, v in dict(group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE, balanced=BALANCED).items():
    if meta.get(k) != v:
        problems.append(f"data option {k}: sample file {meta.get(k)} vs this notebook {v}")
assert not problems, "the sample file does not match this notebook:\n  " + "\n  ".join(problems)
if meta.get("guidance", 1.0) != 1.0:
    print(f"WARNING: guidance {meta['guidance']}; distribution metrics assume 1.0")

gen_mel, gen_labels = s["samples"].float(), s["labels"].long()
if GEN_PER_CLASS is not None:
    keep = torch.cat([torch.nonzero(gen_labels == c).squeeze(1)[:GEN_PER_CLASS] for c in torch.unique(gen_labels)])
    gen_mel, gen_labels = gen_mel[keep], gen_labels[keep]
print(f"generated: {tuple(gen_mel.shape)} | train steps {meta.get('train_steps')} | guidance {meta.get('guidance')} | "
      f"DDIM steps {meta.get('steps')} | commit {meta.get('commit')}")
print("per class:", torch.bincount(gen_labels).tolist())

## 4. Data and the real reference set

The validation fold, center-cropped to the model's clip length (19,812 samples, 1.98 s), the same
crop the front-end probe used. The reference takes `REAL_PER_GEN` times as many clips per class as
the generated set has, drawn at random within each class. Each clip keeps its recording group, so
clips from one recording always stay on the same side of a real-vs-real split.

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
class_names = data["class_names"]
assert class_names == meta.get("class_names", class_names), "class order differs from the sample file"
train_split, val_split, _ = C.split_by_fold(data, cfg)        # fold 5 is not used anywhere in this notebook
del data
n_classes = len(class_names)
fine_idx = [class_names.index(n) for n in FINE]

def center_crop(w):
    off = (w.shape[-1] - cfg.clip_samples) // 2
    return w[..., off:off + cfg.clip_samples]

frontend = C.MelFrontEnd.load(FRONTEND_FILE, cfg, device=device)   # the generator's own front end and dB window
print(f"dB window: [{frontend.db_max - cfg.db_range:.1f}, {frontend.db_max:.1f}]")

rng = np.random.default_rng(cfg.seed)
yval_all = val_split["labels"].numpy()
ref_idx = []
for c in range(n_classes):
    want = REAL_PER_GEN * int((gen_labels == c).sum())
    pool = np.flatnonzero(yval_all == c)
    if want > len(pool):
        print(f"WARNING: {class_names[c]} has only {len(pool)} validation clips; using all of them")
    ref_idx.append(rng.choice(pool, min(want, len(pool)), replace=False))
ref_idx = np.sort(np.concatenate(ref_idx))
real_wave = center_crop(val_split["waveforms"][torch.from_numpy(ref_idx)]).float()
real_labels = val_split["labels"][torch.from_numpy(ref_idx)]
groups = np.array(val_split["groups"])[ref_idx]
print(f"real reference: {len(real_wave)} clips from {len(np.unique(groups))} recording groups "
      f"| per class {torch.bincount(real_labels).tolist()}")

## 5. Learnability: training clips vs validation clips

The relative v-loss from the training notebook: the model's v-prediction error divided by the
error of the best predictor that knows only each mel band's variance (1.0 = learned nothing
beyond that). Measured on the EMA weights, at five noise levels, with **the same noise** for the
training and validation sets so the two numbers are directly comparable.

* **Train** is a measure of fit. ESC-50 Path A at 128 bands scored 0.121.
* **Validation / train** is the generalization gap. Close to 1: the model learned the classes, not
  the particular clips. Well above 1: it fits the training clips much better than new ones.

In [ ]:
ck = torch.load(f"{RUN_DIR}/last.pt", map_location=device)
assert ck["config"]["n_mels"] == cfg.n_mels and ck["config"].get("f_max") == cfg.f_max, "checkpoint was trained with another front end"
print(f"checkpoint: step {ck['step']} of {ck['config']['total_steps']}")
unet = C.build_unet2d(cfg, dropout=0.0).to(device)
ema = EMAModel(unet.parameters(), decay=0.999)
ema.load_state_dict(ck["ema"]); ema.copy_to(unet.parameters()); unet.eval()
del ck
scheduler, _ = C.make_schedulers(cfg)

def balanced_pick(labels, k, seed):
    r = np.random.default_rng(seed)
    labels = np.asarray(labels)
    return np.concatenate([r.choice(np.flatnonzero(labels == c), min(k, int((labels == c).sum())), replace=False)
                           for c in range(n_classes)])

@torch.no_grad()
def relative_vloss(waves, labels, t_values=(100, 300, 500, 700, 900), seed=0, batch=UNET_BATCH):
    # per-class and pooled variance-weighted relative v-loss, same formula as the training notebook
    x0 = torch.cat([frontend(waves[i:i + batch].to(device)) for i in range(0, len(waves), batch)])
    labels = labels.to(device)
    out = {}
    for name, mask in [("all", torch.ones_like(labels, dtype=torch.bool))] + [(c, labels == i) for i, c in enumerate(class_names)]:
        x, y = x0[mask], labels[mask]
        var = x.var(dim=(0, 1, 3)).cpu()
        err, base = torch.zeros(cfg.n_mels), torch.zeros(cfg.n_mels)
        g = torch.Generator(device="cpu").manual_seed(seed)
        for t_val in t_values:
            noise = torch.randn(x.shape, generator=g).to(device)
            t = torch.full((len(x),), t_val, device=device)
            noisy, target = scheduler.add_noise(x, noise, t), scheduler.get_velocity(x, noise, t)
            pred = torch.cat([unet(noisy[i:i + batch], t[i:i + batch], class_labels=y[i:i + batch]).sample for i in range(0, len(x), batch)])
            err += ((pred - target) ** 2).mean(dim=(0, 1, 3)).cpu() / len(t_values)
            ac = scheduler.alphas_cumprod[t_val].item(); a, b = ac ** 0.5, (1 - ac) ** 0.5
            base += ((a**2 + b**2 * var) - (a * b * (1 - var)) ** 2 / (a**2 * var + b**2)) / len(t_values)
        rel = err / base.clamp(min=1e-8)
        out[name] = float((rel * var).sum() / var.sum())
    return out

tr_idx = torch.from_numpy(balanced_pick(train_split["labels"].numpy(), LEARN_PER_CLASS, 1))
va_idx = torch.from_numpy(balanced_pick(yval_all, LEARN_PER_CLASS, 2))
learn_train = relative_vloss(center_crop(train_split["waveforms"][tr_idx]).float(), train_split["labels"][tr_idx])
learn_val = relative_vloss(center_crop(val_split["waveforms"][va_idx]).float(), val_split["labels"][va_idx])
learn = pd.DataFrame({"train": learn_train, "validation": learn_val})
learn["val / train"] = learn["validation"] / learn["train"]
learn.round(3)

## 6. Everything to audio, plus the ceiling

* **Generated:** Path A's mel spectrograms through Griffin-Lim (the model has no vocoder).
* **Ceiling:** the real reference clips through the same front end and the same Griffin-Lim, no
  generator. It is the best any generator in this representation could score. Its rows line up
  one-to-one with the real reference.

Both are cached in `EVAL_DIR` and recomputed only when the sample file changes.

In [ ]:
def cached(key, source_file, fn):
    path = EVAL_DIR / f"{key}.pt"
    stamp = os.path.getmtime(source_file) if source_file else 0.0
    if path.exists():
        d = torch.load(path)
        if d["stamp"] == stamp and d.get("n") == d["value"].shape[0]:
            return d["value"]
    value = fn()
    torch.save({"value": value, "stamp": stamp, "n": value.shape[0]}, path)
    return value

def to_mel(w, batch=64):
    return torch.cat([frontend(w[i:i + batch].to(device)).cpu() for i in range(0, len(w), batch)])

gen_wave = cached(f"gen_wave_gl{GL_ITERS}", SAMPLES_FILE, lambda: C.spec_to_wave(gen_mel, frontend, n_iter=GL_ITERS, seed=cfg.seed))
real_mel = to_mel(real_wave)
ceil_wave = cached(f"ceil_wave_gl{GL_ITERS}_n{len(real_wave)}", None, lambda: C.spec_to_wave(real_mel, frontend, n_iter=GL_ITERS, seed=cfg.seed))
sets = {"real": real_wave, "ceiling": ceil_wave, "generated": gen_wave}
set_labels = {"real": real_labels, "ceiling": real_labels, "generated": gen_labels}
for k, w in sets.items():
    print(f"{k:<10}{tuple(w.shape)}")

## 7. Look and listen

**Average spectrum per class.** Each clip's log-mel spectrum averaged over time, then the median
and interquartile range over clips. This is the direct check that the generator puts each species'
energy where the real calls are: groupers around 80-140 Hz, squirrelfish with a band near
750-1,100 Hz, vessels with tonals at 250-1,000 Hz, toadfish lowest. Compared in mel space, since
that is what the generator produces; Griffin-Lim is judged separately by the ceiling.

In [ ]:
fb = frontend.mel.mel_scale.fb                                       # (n_freqs, n_mels)
band_hz = (fb.argmax(dim=0).float() * cfg.sample_rate / cfg.n_fft).clamp(min=cfg.f_min).cpu().numpy()
gen_mel_c = gen_mel.clamp(-1, 1)
spec_real = real_mel[:, 0].mean(dim=-1).numpy()                     # (N, n_mels), per-clip average spectrum
spec_gen = gen_mel_c[:, 0].mean(dim=-1).numpy()
db = lambda x: (x + 1) / 2 * cfg.db_range + frontend.db_max - cfg.db_range   # back to dB

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for c, ax in enumerate(axes.flat):
    for spec, lab, col in [(spec_real, real_labels, "#555555"), (spec_gen, gen_labels, "#3b6ea8")]:
        x = db(spec[lab.numpy() == c])
        q1, med, q3 = np.percentile(x, [25, 50, 75], axis=0)
        ax.plot(band_hz, med, color=col, label="real" if col == "#555555" else "generated")
        ax.fill_between(band_hz, q1, q3, color=col, alpha=0.2)
    ax.set_xscale("log"); ax.set_title(class_names[c]); ax.grid(alpha=0.3)
    for f in (250, 1000):
        ax.axvline(f, color="k", lw=0.5, ls=":")
axes[0, 0].legend(); axes[1, 0].set_xlabel("Hz"); axes[1, 1].set_xlabel("Hz"); axes[1, 2].set_xlabel("Hz")
axes[0, 0].set_ylabel("dB"); axes[1, 0].set_ylabel("dB")
plt.suptitle("time-averaged mel spectrum: median and IQR over clips"); plt.tight_layout()
plt.savefig(EVAL_DIR / "class_spectra.png", dpi=130); plt.show()

In [ ]:
fig, axes = plt.subplots(4, n_classes, figsize=(2.8 * n_classes, 9))
for c in range(n_classes):
    r = torch.nonzero(real_labels == c).squeeze(1)[:2]
    g = torch.nonzero(gen_labels == c).squeeze(1)[:2]
    for row, img in enumerate([real_mel[r[0], 0], real_mel[r[1], 0], gen_mel_c[g[0], 0], gen_mel_c[g[1], 0]]):
        axes[row, c].imshow(img, origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
    axes[0, c].set_title(class_names[c], fontsize=9)
for row, t in enumerate(["real", "real", "generated", "generated"]):
    axes[row, 0].set_ylabel(t)
plt.tight_layout(); plt.show()

for c in range(n_classes):
    i = int(torch.nonzero(real_labels == c)[0]); j = torch.nonzero(gen_labels == c).squeeze(1)[:2]
    print(f"=== {class_names[c]}: real, real through mel + Griffin-Lim (ceiling), two generated")
    for w in [real_wave[i, 0], ceil_wave[i, 0], gen_wave[j[0], 0], gen_wave[j[1], 0]]:
        display(Audio(w.numpy(), rate=cfg.sample_rate, normalize=True))

**Class similarity, real vs generated.** Ali's question about classes 2 & 8 and 4 & 7 was how
closely their spectrograms resemble each other within the same frequency ranges. Here: the
correlation between classes' median spectra, per range, for real and for generated clips. If the
generator keeps the structure, the two matrices should look alike; if it makes classes more
similar than they really are, that shows up as higher off-diagonal values.

In [ ]:
RANGES = {"all": (cfg.f_min, cfg.sample_rate / 2), "<250 Hz": (cfg.f_min, 250), "250-1000 Hz": (250, 1000)}
def class_corr(spec, lab, lo, hi):
    m = (band_hz >= lo) & (band_hz < hi)
    med = np.stack([np.median(spec[lab.numpy() == c][:, m], axis=0) for c in range(n_classes)])
    return np.corrcoef(med)

similarity = {}
fig, axes = plt.subplots(2, len(RANGES), figsize=(5 * len(RANGES), 9))
for j, (rname, (lo, hi)) in enumerate(RANGES.items()):
    for i, (tag, spec, lab) in enumerate([("real", spec_real, real_labels), ("generated", spec_gen, gen_labels)]):
        cm = class_corr(spec, lab, lo, hi); similarity[f"{tag} {rname}"] = cm
        ax = axes[i, j]; ax.imshow(cm, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_xticks(range(n_classes), class_names, rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(n_classes), class_names, fontsize=7); ax.set_title(f"{tag}, {rname}", fontsize=9)
        for a in range(n_classes):
            for b in range(n_classes):
                ax.text(b, a, f"{cm[a, b]:.2f}", ha="center", va="center", fontsize=6)
plt.tight_layout(); plt.show()

pairs = [("Nassau grouper", "Toadfish"), ("Black grouper", "Vessel")]       # Ali's "2 and 8", "4 and 7"
for a, b in pairs:
    ia, ib = class_names.index(a), class_names.index(b)
    print(f"{a} vs {b}: " + " | ".join(f"{r}: real {similarity['real ' + r][ia, ib]:.2f}, generated {similarity['generated ' + r][ia, ib]:.2f}" for r in RANGES))

## 8. Level check

Real clips were normalized to the same RMS (-26 dBFS) before training. A generator that learned
the data should produce clips at about the same level. Griffin-Lim output is not clipped, so a
peak above 1 means the phase reconstruction overshoots.

In [ ]:
def rms_db(w):
    return 20 * torch.log10(w.pow(2).mean(dim=(1, 2)).sqrt().clamp(min=1e-6))
print(f"{'set':<11}{'median dBFS':>12}{'IQR':>18}{'median peak':>13}{'peak > 1':>10}")
for name, w in sets.items():
    r, peaks = rms_db(w), w.abs().amax(dim=(1, 2))
    print(f"{name:<11}{r.median().item():>12.1f}{f'[{r.quantile(0.25).item():.1f}, {r.quantile(0.75).item():.1f}]':>18}"
          f"{peaks.median().item():>13.3f}{(peaks > 1).float().mean().item():>10.1%}")

## 9. The judge classifier: class fidelity

One small CNN trained on real clips from the training folds, with the probe's recipe and the
generator's own front end. It is used twice: here as a **judge** (does a generated Nassau grouper
get recognised as a Nassau grouper?) and in section 10 as the **in-domain embedder**.

| Choice | Reason |
|---|---|
| The probe network: four blocks of two 3x3 convolutions + batch norm + ReLU (32, 64, 128, 256 channels), 2x2 max-pooling, average over time, keep the 8 frequency rows, dropout 0.3, one linear layer | The probe showed it separates the six classes at 0.96-0.98 F1 on real validation clips, the three groupers included. Averaging over time makes it insensitive to where a call starts in the window; keeping the frequency axis lets it use where the energy sits, which is what separates the groupers. |
| Trained on real audio only (folds 1-3), never on generated or Griffin-Lim audio | A judge that has seen synthetic audio would learn its artefacts. Trained on real clips only, its accuracy on generated clips measures how real-like they are, class by class. |
| The generator's own front end and dB window | The judge then sees generated audio exactly as the generator's training data looked, so it measures the generator, not a mismatch between front ends. |
| One fixed seed, saved and reused | The same judge has to score every synthetic set that is evaluated later (other checkpoints, guidance values, TSTR sets); otherwise differences between sets would mix with differences between judges. |
| Class-balanced sampling, AdamW + one-cycle, fixed 6 epochs, no early stopping | The probe's recipe unchanged. No early stopping because the validation fold is the evaluation set. |
| Embedding = the 2,048 values before the output layer (256 channels x 8 frequency rows) | The layer that carries the class information the judge relies on. Distances there emphasise exactly what distinguishes the species, which PANNs and CLAP were never trained to notice. |

The comparison that matters is **generated vs ceiling**: the ceiling (real clips after
Griffin-Lim) shows how well a perfect generator in this representation could score. The table also
shows the generated mel spectrograms scored directly, without Griffin-Lim, which separates the
generator's errors from the inversion's.

In [ ]:
class ProbeCNN(nn.Module):
    def __init__(self, n_classes, n_bands=128, width=(32, 64, 128, 256), p_drop=0.3):
        super().__init__()
        layers, c_in = [], 1
        for c in width:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(width[-1] * (n_bands // 2 ** len(width)), n_classes))

    def embed(self, x):                        # (B, 1, n_bands, frames) -> (B, 2048)
        return self.features(x).mean(dim=-1).flatten(1)

    def forward(self, x):
        return self.head(self.embed(x))


JUDGE_FILE = Path(cfg.out_root) / "eval" / f"judge{TAG}_seed{JUDGE_SEED}_ep{EPOCHS}.pt"
judge = ProbeCNN(n_classes).to(device)
if JUDGE_FILE.exists() and not QUICK_TEST:
    judge.load_state_dict(torch.load(JUDGE_FILE, map_location=device)); print("loaded", JUDGE_FILE)
else:
    t0 = time.time()
    train_wave = center_crop(train_split["waveforms"]).to(device)          # float16 on the GPU
    ytr = train_split["labels"].to(device)
    counts = torch.bincount(train_split["labels"], minlength=n_classes).float()
    w_train = (1.0 / counts)[train_split["labels"]]
    torch.manual_seed(JUDGE_SEED)
    judge = ProbeCNN(n_classes).to(device)
    steps = EPOCHS * (len(ytr) // BATCH)
    opt = torch.optim.AdamW(judge.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    g = torch.Generator().manual_seed(JUDGE_SEED)
    judge.train()
    for step in range(steps):
        idx = torch.multinomial(w_train, BATCH, replacement=True, generator=g).to(device)
        x = frontend(train_wave[idx].float())                                 # front end in float32, as in the probe
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            loss = F.cross_entropy(judge(x), ytr[idx])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    del train_wave
    if not QUICK_TEST:
        JUDGE_FILE.parent.mkdir(parents=True, exist_ok=True); torch.save(judge.state_dict(), JUDGE_FILE)
    print(f"trained the judge in {(time.time() - t0) / 60:.1f} min")
judge.eval()

@torch.no_grad()
def judge_run(x, is_mel=False, batch=256):
    # class probabilities and embeddings for audio (or, with is_mel, for mel spectrograms in [-1, 1])
    probs, embs = [], []
    for i in range(0, len(x), batch):
        m = x[i:i + batch].to(device).float().clamp(-1, 1) if is_mel else frontend(x[i:i + batch].to(device).float())
        e = judge.embed(m)
        probs.append(judge.head(e).softmax(-1).cpu()); embs.append(e.cpu())
    return torch.cat(probs).numpy(), torch.cat(embs).double().numpy()

judged = {name: judge_run(w) for name, w in sets.items()}
judged["generated (mel, no Griffin-Lim)"] = judge_run(gen_mel, is_mel=True)

def recall_per_class(y, p):
    return np.array([(p[y == c] == c).mean() if (y == c).any() else np.nan for c in range(n_classes)])

rows = []
for name, (probs, _) in judged.items():
    y = (gen_labels if name.startswith("generated") else real_labels).numpy()
    rec = recall_per_class(y, probs.argmax(1))
    rows.append({"set": name, "mean recall": np.nanmean(rec), "fine-grained": np.nanmean(rec[fine_idx]),
                 **{n: rec[i] for i, n in enumerate(class_names)}})
fidelity = pd.DataFrame(rows).set_index("set")
fidelity.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name in zip(axes, ["ceiling", "generated"]):
    y = set_labels[name].numpy(); p = judged[name][0].argmax(1)
    cm = np.bincount(y * n_classes + p, minlength=n_classes ** 2).reshape(n_classes, n_classes)
    cm = cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(n_classes), class_names, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(n_classes), class_names, fontsize=7); ax.set_title(f"judge on {name} audio (rows: intended class)", fontsize=9)
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", color="w", fontsize=7)
plt.tight_layout(); plt.savefig(EVAL_DIR / "judge_confusion.png", dpi=130); plt.show()

## 10. Embeddings

* **probe**: the judge's 2,048-d embedding (in-domain, primary).
* **PANNs CNN14** (AudioSet tagger, 2,048-d, runs at 32 kHz).
* **LAION-CLAP** (`laion/clap-htsat-unfused`, 512-d, runs at 48 kHz).

Audio is upsampled from 10 kHz for PANNs and CLAP. Nothing above 5 kHz exists in any set, real or
generated, so the upsampling treats every set the same; it does mean both models see only the
lowest part of the range they were trained on.

In [ ]:
def load_panns():
    from panns_inference import AudioTagging
    tagger = AudioTagging(checkpoint_path=None, device=device)
    resample = torchaudio.transforms.Resample(cfg.sample_rate, 32000)
    return lambda w: tagger.inference(resample(w[:, 0]).numpy())[1]

def load_clap():
    from transformers import ClapModel, ClapFeatureExtractor
    model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
    fx = ClapFeatureExtractor.from_pretrained("laion/clap-htsat-unfused")
    resample = torchaudio.transforms.Resample(cfg.sample_rate, 48000)
    @torch.no_grad()
    def embed(w):
        audio = [a.numpy() for a in resample(w[:, 0])]
        inputs = {k: v.to(device) for k, v in fx(audio, sampling_rate=48000, truncation="rand_trunc", padding="repeatpad", return_tensors="pt").items()}
        out = model.get_audio_features(**inputs)
        return (out.pooler_output if hasattr(out, "pooler_output") else out).cpu().numpy()
    return embed

LOADERS = {"panns": load_panns, "clap": load_clap}

def embed_all(embedder, batch=32):
    if embedder == "probe":
        return {name: judged[name][1] for name in sets}
    fn, out = None, {}
    for name, w in sets.items():
        path = EVAL_DIR / f"emb_{embedder}_{name}_n{len(w)}.npy"
        stamp = os.path.getmtime(SAMPLES_FILE) if name == "generated" else 0.0
        if path.exists() and path.with_suffix(".stamp").exists() and float(path.with_suffix(".stamp").read_text()) == stamp:
            out[name] = np.load(path); continue
        fn = fn or LOADERS[embedder]()
        out[name] = np.concatenate([fn(w[i:i + batch]) for i in range(0, len(w), batch)]).astype(np.float64)
        np.save(path, out[name]); path.with_suffix(".stamp").write_text(str(stamp))
    return out

emb = {e: embed_all(e) for e in EMBEDDERS}
for e, d in emb.items():
    print(e, {k: v.shape for k, v in d.items()})

## 11. Main result: all classes together

`M.evaluate` against the real reference, as on ESC-50:

* **ratio**: Frechet distance divided by the real-vs-real floor at the same sample size
  (1 = indistinguishable from a second draw of real data).
* **above**: fraction of the splits where the set was farther from real than real was from itself.
* **repr. share**: the part of the generated set's excess over the floor already present in the
  ceiling, that is, what Griffin-Lim and the mel representation cost before the generator does
  anything.
* **mean / cov**: how the generated set's distance divides between a shift of the average sound
  and a difference in spread or shape.

In [ ]:
def summarize(res, e, tag="all"):
    rep = res["report"]; g, c = rep["generated"]["w2_sq"], rep["ceiling"]["w2_sq"]
    excess = g["value"] - g["floor"]
    ms, cs = rep["generated"]["mean_sq"]["value"], rep["generated"]["cov_sq"]["value"]
    return dict(embedder=e, classes=tag, ceiling_ratio=c["ratio"], generated_ratio=g["ratio"], generated_above=g["frac_above"],
                representation_share=(c["value"] - c["floor"]) / excess if excess > 0 else float("nan"),
                mean_share=ms / (ms + cs) if (ms + cs) > 0 else float("nan"),
                mmd_p=res["mmd"]["generated"]["p_value"], off_subspace=res["off_subspace"]["generated"],
                contraction=res["spectra"]["generated"].contraction_share, log_volume=res["spectra"]["generated"].log_volume_ratio)

results, summary = {}, []
for e in EMBEDDERS:
    d = emb[e]
    res = M.evaluate(d["real"], {"ceiling": d["ceiling"], "generated": d["generated"]}, groups_real=groups,
                     aligned=["ceiling"], max_dim=min(MAX_DIM, len(d["real"]) // 8), standardize=True,
                     n_splits=N_SPLITS, n_perm=N_PERM, seed=cfg.seed)
    results[e] = res
    print(f"\n==================== {e} ====================")
    print(M.format_report(res["report"]))
    summary.append(summarize(res, e))

overall = pd.DataFrame(summary).set_index("embedder")
overall[["ceiling_ratio", "generated_ratio", "generated_above", "representation_share", "mean_share", "mmd_p", "off_subspace"]].round(3)

## 12. Per class

The same comparison one class at a time: each class's generated clips against that class's real
clips, with that class's own floor and ceiling and a PCA fitted on that class only. With about 100
clips per half the numbers are noisier than the overall ones, so `MAX_DIM_CLASS` is small. A class
whose generated ratio is far above the others is where the generator is weakest.

In [ ]:
per_class = []
for e in EMBEDDERS:
    d = emb[e]
    for c, cname in enumerate(class_names):
        mr, mg = real_labels.numpy() == c, gen_labels.numpy() == c
        res = M.evaluate(d["real"][mr], {"ceiling": d["ceiling"][mr], "generated": d["generated"][mg]}, groups_real=groups[mr],
                         aligned=["ceiling"], max_dim=min(MAX_DIM_CLASS, max(2, int(mr.sum()) // 8)), standardize=True,
                         n_splits=N_SPLITS, n_perm=min(N_PERM, 200), seed=cfg.seed)
        per_class.append(summarize(res, e, cname))
per_class = pd.DataFrame(per_class)
table = per_class.pivot(index="classes", columns="embedder", values=["ceiling_ratio", "generated_ratio"]).reindex(class_names)
table.round(2)

In [ ]:
fig, axes = plt.subplots(1, len(EMBEDDERS), figsize=(5.5 * len(EMBEDDERS), 3.8), squeeze=False)
for ax, e in zip(axes[0], EMBEDDERS):
    rows = per_class[per_class.embedder == e].set_index("classes").reindex(class_names)
    x = np.arange(n_classes)
    ax.bar(x - 0.2, rows["ceiling_ratio"], 0.4, label="ceiling (representation only)", color="#9aa5b1")
    ax.bar(x + 0.2, rows["generated_ratio"], 0.4, label="generated", color="#3b6ea8")
    ax.axhline(1.0, color="black", lw=1, ls="--"); ax.set_yscale("log")
    ax.set_xticks(x, class_names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Frechet / real-vs-real floor"); ax.set_title(f"{e}, per class")
axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(EVAL_DIR / "per_class_frechet.png", dpi=130); plt.show()

## 13. Spread: does the generator cover the real variation?

The stretch spectrum (primary embedder, all classes) splits the covariance term into independent
directions. Below 1: too little variation along that direction, i.e. samples concentrated on the
most typical sounds. Above 1: too much (noise, artefacts). The grey curve is real data against
itself. On ESC-50 every path was under-dispersed; the per-class columns show whether that holds
here too, and for which species.

In [ ]:
e = EMBEDDERS[0]
Zr = results[e]["reducer"].transform(emb[e]["real"])
ug = np.unique(groups)
half = np.isin(groups, np.random.default_rng(cfg.seed).permutation(ug)[: len(ug) // 2])
ref = M.stretch_spectrum(Zr[half], Zr[~half])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.sort(ref.stretch), color="gray", lw=2, label="real vs real")
for name in ("ceiling", "generated"):
    ax.plot(np.sort(results[e]["spectra"][name].stretch), marker="o", ms=3, label=name)
ax.axhline(1.0, color="gray", lw=1, ls="--"); ax.set_yscale("log")
ax.set_xlabel("direction (sorted by stretch)"); ax.set_ylabel("stretch (1 = matches real)")
ax.set_title(f"{e}: below 1 too little variation, above 1 too much"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
per_class[per_class.embedder == e].set_index("classes").reindex(class_names)[["contraction", "log_volume"]].round(2)

## 14. Memorization check

For every generated clip, the distance (in the generator's own mel space) to its nearest
training clip, against the same distance for unseen real validation clips. Real unseen clips show
how close "new but similar" sits to the training set. If generated clips sit clearly closer, the
model is reproducing training clips rather than generating new ones. The last column counts
generated clips closer to the training set than 99% of validation clips are, which is where copies
would show up.

In [ ]:
@torch.no_grad()
def train_bank(batch=256):
    w = train_split["waveforms"]
    return torch.cat([frontend(center_crop(w[i:i + batch]).to(device).float()).half() for i in range(0, len(w), batch)])

bank = train_bank()
nn_gen = C.nearest_neighbor_distance(gen_mel_c.to(device), bank).cpu().numpy()
nn_val = C.nearest_neighbor_distance(real_mel.to(device), bank).cpu().numpy()
del bank
thr = np.percentile(nn_val, 1)
rows = []
for c, cname in enumerate(class_names):
    g, v = nn_gen[gen_labels.numpy() == c], nn_val[real_labels.numpy() == c]
    rows.append({"class": cname, "median NN, validation": np.median(v), "median NN, generated": np.median(g),
                 "generated / validation": np.median(g) / np.median(v), "generated below val 1%": (g < thr).mean()})
memo = pd.DataFrame(rows).set_index("class")
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(nn_val, bins=50, alpha=0.6, density=True, label="validation -> train")
ax.hist(nn_gen, bins=50, alpha=0.6, density=True, label="generated -> train")
ax.set_xlabel("distance to nearest training clip (mel space)"); ax.legend(); plt.tight_layout(); plt.show()
memo.round(3)

## 15. Save

In [ ]:
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
out = {
    "commit": commit, "samples_file": SAMPLES_FILE, "run_dir": RUN_DIR, "quick_test": QUICK_TEST,
    "sample_meta": {k: v for k, v in meta.items() if k != "data_report"},
    "reference": {"fold": cfg.val_fold, "n": len(real_wave), "per_class": torch.bincount(real_labels).tolist()},
    "learnability": learn.round(4).to_dict(),
    "fidelity": fidelity.round(4).reset_index().to_dict("records"),
    "overall": overall.round(4).reset_index().to_dict("records"),
    "per_class": per_class.round(4).to_dict("records"),
    "memorization": memo.round(4).reset_index().to_dict("records"),
    "similarity": {k: np.round(v, 3).tolist() for k, v in similarity.items()},
    "reports": {e: r["report"] for e, r in results.items()},
}
with open(EVAL_DIR / f"results_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved", EVAL_DIR / f"results_{commit}.json")

## How to read the result

* **Start with class fidelity (section 9).** It answers the most practical question: does each
  generated class still sound like that class to a classifier trained on real clips? Read the
  generated row against the ceiling row, not against 1.0. Generated close to the ceiling: the
  generator kept the class information, and what is lost comes from Griffin-Lim. Generated far
  below on one class: that class is where synthetic training data would mislead a classifier.
  Vessel and toadfish will score high regardless, because their recording sites give them away.
* **Then the per-class distances (section 12).** Same reading as on ESC-50: the ceiling is the
  representation's cost, generated minus ceiling is the generator's. A class with a large
  generator gap is a candidate for more training or a different checkpoint.
* **The embedders should agree on the direction.** The probe embedder is the most relevant for
  this data; PANNs and CLAP are the standard references. If they disagree, report it.
* **Learnability and memorization are sanity checks.** A validation/train ratio well above 1 or
  generated clips sitting closer to the training set than unseen clips are means the model is
  fitting clips, not classes, and the synthetic set would add little new information.
* **This is still a gate, not the thesis question.** Whether these clips are *useful* is answered
  by TSTR on fold 5. The numbers here (fidelity, per-class ratios, spread) are the predictors the
  TSTR study will test.